In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','features_cic']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, features_cic as fc
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [6]:
# =============================================================================
# Cell 2 - WHICH REALIZATIONS TO TRAIN THIS RUN. Leave all five for one long
# run, or set a subset to run in chunks. Already-trained models are skipped, so
# the notebook is resumable and safe to interrupt.
# =============================================================================
REALIZATIONS_TO_RUN = [
    'R1_holdout_Slowhttptest',
    'R2_holdout_Slowloris',
    'R3_holdout_GoldenEye',
    'R4_holdout_Slowloris_Slowhttptest',
    'R5_holdout_GoldenEye_Slowloris',
]
CLASSES = ['Benign', 'DoS']          # column order for all saved probabilities; index 0=Benign, 1=DoS

cic = pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed = cic[cic['day']=='wednesday'].reset_index(drop=True)
wed = wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
FCOLS = fc.feature_cols(wed)

PROBS_DIR = config.DATA_DIR / 'cic_probs'; PROBS_DIR.mkdir(parents=True, exist_ok=True)
print('Wednesday rows:', len(wed), '| features:', len(FCOLS),
      '| realizations this run:', len(REALIZATIONS_TO_RUN))


Wednesday rows: 477860 | features: 81 | realizations this run: 5


In [7]:
# =============================================================================
# Cell 3 - fixed hyperparameters (logged deviation), models on INTEGER labels
# (Benign 0, DoS 1), and the isotonic OvR calibrator (preregistration 6).
# Integer labels avoid the sklearn MLP early-stopping isnan-on-strings bug and
# give a uniform [Benign, DoS] probability column order.
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score
import xgboost as xgb

HP = {
  'rf':  dict(n_estimators=200, min_samples_leaf=2, n_jobs=-1),
  'xgb': dict(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist',
              n_jobs=-1, eval_metric='logloss'),
  'mlp': dict(hidden_layer_sizes=(128,64), early_stopping=True, max_iter=60, batch_size=512),
}

def train_one(arch, Xtr, ytr_int, scal):
    if arch == 'rf':
        m = RandomForestClassifier(random_state=SEED, **HP['rf']); m.fit(Xtr, ytr_int)
    elif arch == 'xgb':
        m = xgb.XGBClassifier(random_state=SEED, **HP['xgb']); m.fit(Xtr, ytr_int)
    else:
        m = MLPClassifier(random_state=SEED, **HP['mlp']); m.fit(scal.transform(Xtr), ytr_int)
    return m

def prob2(m, arch, X, scal):
    Xin = scal.transform(X) if arch == 'mlp' else X
    P = m.predict_proba(Xin); cls = list(m.classes_)     # classes_ == [0, 1]
    return np.c_[P[:, cls.index(0)], P[:, cls.index(1)]]  # columns [Benign, DoS]

def isotonic_ovr(prob_cal, y_int, prob_apply):
    out = np.zeros_like(prob_apply)
    for j in range(prob_apply.shape[1]):
        ir = IsotonicRegression(out_of_bounds='clip')
        ir.fit(prob_cal[:, j], (y_int == j).astype(float))
        out[:, j] = ir.predict(prob_apply[:, j])
    s = out.sum(axis=1, keepdims=True); s[s == 0] = 1.0
    return out / s

print('model + calibration helpers ready (integer labels)')


model + calibration helpers ready (integer labels)


In [10]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
import config, features_cic as fc

cic = pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed = cic[cic['day']=='wednesday'].reset_index(drop=True)
wed = wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
FCOLS = fc.feature_cols(wed)
y = (wed['label']=='DoS').astype(int).to_numpy()

print('n features:', len(FCOLS))
print('\nfeature columns:')
print(FCOLS)

rows = []
for c in FCOLS:
    x = pd.to_numeric(wed[c], errors='coerce').to_numpy(dtype=float)
    x = np.where(np.isfinite(x), x, np.nan)
    if np.isnan(x).all() or np.nanstd(x[~np.isnan(x)]) == 0:
        auc = 0.5
    else:
        med = np.nanmedian(x); xx = np.where(np.isnan(x), med, x)
        auc = roc_auc_score(y, xx)
    rows.append({'feature': c, 'auc': round(float(auc), 4), 'separation': round(abs(auc - 0.5), 4)})

imp = pd.DataFrame(rows).sort_values('separation', ascending=False)
print('\nsingle-feature separability (auc near 1.0 or 0.0 = one feature splits it almost perfectly):')
print(imp.head(20).to_string(index=False))

n features: 81

feature columns:
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag 

In [ ]:
# =============================================================================
# Cell 4 - RESUMABLE training. For each (realization, seed, arch): train on the
# source train partition, isotonic-calibrate on probcal, save calibrated probs
# on the source calibration pool and full target. Skips any already saved.
# =============================================================================
def strat(df, fr, seed):
    rng=np.random.default_rng(seed); names=list(fr); f=np.array([fr[k] for k in names],float)
    big=names[int(np.argmax(f))]; a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby('label',sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(f*n).astype(int); c[names.index(big)]+=n-c.sum(); k=0
        for nm,q in zip(names,c): a.loc[idx[k:k+q]]=nm; k+=q
    return a

variant_map = {'Slowhttptest':'DoS Slowhttptest','Slowloris':'DoS Slowloris','GoldenEye':'DoS GoldenEye'}
perf = []; t0 = time.time()
for name in REALIZATIONS_TO_RUN:
    srcpool_idx = np.load(config.PROC_DIR / f'cic_{name}_srcpool_idx.npy')
    target_idx  = np.load(config.PROC_DIR / f'cic_{name}_target_idx.npy')
    held = [variant_map[h] for h in name.split('holdout_')[1].split('_') if h in variant_map]
    kept = [v for v in ['DoS Hulk','DoS GoldenEye','DoS Slowloris','DoS Slowhttptest'] if v not in held]

    benign = wed[wed.label=='Benign']
    rng_b = np.random.default_rng(777); bp = benign.index.to_numpy().copy(); rng_b.shuffle(bp)
    BEN_SRC = set(bp[:int(len(bp)*0.50)])
    dos_src = wed[(wed.label=='DoS') & (wed.subtype.isin(kept))]
    src = pd.concat([dos_src, benign.loc[sorted(BEN_SRC)]]).copy()
    seed_name = int(hashlib.sha256(name.encode()).hexdigest(),16) % (2**31)
    src = src.assign(partition=strat(src, config.SPLIT_FRACTIONS, seed_name).values)
    Dtr, Dva, Dpc = src[src.partition=='train'], src[src.partition=='val'], src[src.partition=='probcal']

    imp = SimpleImputer(strategy='median').fit(fc.matrix(Dtr, FCOLS))
    Xtr = imp.transform(fc.matrix(Dtr, FCOLS)); ytr = (Dtr['label']=='DoS').astype(int).to_numpy()
    Xva = imp.transform(fc.matrix(Dva, FCOLS)); yva = (Dva['label']=='DoS').astype(int).to_numpy()
    Xpc = imp.transform(fc.matrix(Dpc, FCOLS)); ypc = (Dpc['label']=='DoS').astype(int).to_numpy()
    Xsp = imp.transform(fc.matrix(wed.loc[srcpool_idx], FCOLS))
    Xtg = imp.transform(fc.matrix(wed.loc[target_idx],  FCOLS))
    scal = StandardScaler().fit(Xtr)

    for SEED in config.SEEDS:
        for arch in ['rf','xgb','mlp']:
            out = PROBS_DIR / f'{name}__{arch}__seed{SEED}.npz'
            if out.exists(): continue
            m = train_one(arch, Xtr, ytr, scal)
            f1 = f1_score(yva, (prob2(m,arch,Xva,scal)[:,1] >= 0.5).astype(int), average='macro')
            pc = prob2(m, arch, Xpc, scal)
            sp_cal = isotonic_ovr(pc, ypc, prob2(m,arch,Xsp,scal))
            tg_cal = isotonic_ovr(pc, ypc, prob2(m,arch,Xtg,scal))
            np.savez_compressed(out, srcpool=sp_cal, target=tg_cal, classes=np.array(CLASSES))
            perf.append({'realization':name,'seed':SEED,'arch':arch,'val_macro_f1':round(float(f1),4)})
            print(f'{name:34s} {arch:4s} seed {SEED:5d}  valF1={f1:.3f}  [{(time.time()-t0)/60:.1f} min]')

print(f'\ndone this run in {(time.time()-t0)/60:.1f} min; models trained now: {len(perf)}')


R4_holdout_Slowloris_Slowhttptest  rf   seed  2024  valF1=1.000  [2.0 min]
R4_holdout_Slowloris_Slowhttptest  xgb  seed  2024  valF1=1.000  [2.4 min]
R4_holdout_Slowloris_Slowhttptest  mlp  seed  2024  valF1=1.000  [2.9 min]
R4_holdout_Slowloris_Slowhttptest  rf   seed     7  valF1=1.000  [4.6 min]
R4_holdout_Slowloris_Slowhttptest  xgb  seed     7  valF1=1.000  [4.9 min]
R4_holdout_Slowloris_Slowhttptest  mlp  seed     7  valF1=1.000  [5.5 min]
R4_holdout_Slowloris_Slowhttptest  rf   seed    91  valF1=1.000  [7.3 min]
R4_holdout_Slowloris_Slowhttptest  xgb  seed    91  valF1=1.000  [7.6 min]
R4_holdout_Slowloris_Slowhttptest  mlp  seed    91  valF1=1.000  [8.1 min]
R4_holdout_Slowloris_Slowhttptest  rf   seed   512  valF1=1.000  [9.9 min]
R4_holdout_Slowloris_Slowhttptest  xgb  seed   512  valF1=1.000  [10.2 min]
R4_holdout_Slowloris_Slowhttptest  mlp  seed   512  valF1=1.000  [10.8 min]
R4_holdout_Slowloris_Slowhttptest  rf   seed  6021  valF1=1.000  [12.6 min]
R4_holdout_Slowloris_S

In [ ]:
# =============================================================================
# Cell 5 - record performance, log the fixed-HP deviation, commit.
# =============================================================================
if perf:
    pf = pd.DataFrame(perf); ppath = config.REPORTS_DIR / 'model_performance_cicids2017.csv'
    if ppath.exists():
        pf = pd.concat([pd.read_csv(ppath), pf]).drop_duplicates(['realization','seed','arch'], keep='last')
    pf.to_csv(ppath, index=False)
    print(pf.groupby(['realization','arch'])['val_macro_f1'].mean().round(3).to_string())

trained = sorted(p.name for p in PROBS_DIR.glob('*.npz'))
(config.REPORTS_DIR / 'models_manifest_cicids2017.json').write_text(json.dumps({
    'environment':'wednesday_within_day','classes':CLASSES,
    'architectures':['rf','xgb','mlp'],'seeds':config.SEEDS,'hyperparameters_fixed':HP,
    'calibration':'one-vs-rest isotonic on probcal, renormalised (section 6)',
    'n_prob_files_present':len(trained),'n_expected_when_complete':5*len(config.SEEDS)*3}, indent=2))

dev = config.REPORTS_DIR / 'deviations.md'
note = ('\n## nb12 - CIC used architecture-appropriate FIXED hyperparameters (RF/XGB/MLP) chosen for '
        'dataset scale, not a per-dataset macro-F1 grid (section 5), to bound compute. Before any coverage.\n')
if dev.exists() and 'nb12 - CIC used architecture-appropriate FIXED' not in dev.read_text():
    with open(dev,'a') as f: f.write(note)

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,d in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,d)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m',f'nb12: CIC 2-class model panel + isotonic calibration ({len(trained)}/{5*len(config.SEEDS)*3} prob files)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
print(f'\nprob files present: {len(trained)} / {5*len(config.SEEDS)*3}. Re-run remaining realizations until complete.')
